# 🤟 Wasel v4 Pro — Gemini AI Live Translator

### الخطوات:
1. شغّل **Cell 1** — تثبيت
2. شغّل **Cell 2** — الصق مفتاح Gemini → الكاميرا تبدأ تلقائياً!

> المفتاح مجاناً من [aistudio.google.com/apikey](https://aistudio.google.com/apikey)

In [ ]:
#@title 📦 Cell 1: Install
!pip install -q "google-genai>=1.0.0" "flask" "flask-cors" "pyngrok" "Pillow"
print("✅ Ready! Run Cell 2.")

In [ ]:
#@title 🎥 Cell 2: Launch
#@markdown Gemini API Key:
GEMINI_API_KEY = "" #@param {type:"string"}
#@markdown Ngrok Auth Token (get free from ngrok.com/signup):
NGROK_TOKEN = "" #@param {type:"string"}

import threading, time, base64, io, json
from flask import Flask, request, jsonify, Response
from flask_cors import CORS
from PIL import Image
import google.genai as genai
from google.genai import types
from pyngrok import ngrok

if not GEMINI_API_KEY:
    raise ValueError("❌ Paste Gemini API key!")

# ─── Gemini ───
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash"
test = client.models.generate_content(model=MODEL, contents="Say OK")
print(f"✅ Gemini: {test.text.strip()}")

PROMPT = """You are a sign language interpreter. 
If you see a hand gesture or sign: reply ONLY the meaning (1-3 words).
If no gesture: reply ...
Examples: Hello, Thank you, Yes, No, Help, I love you, Peace
No explanations. Just the word."""

# ─── Flask API ───
app = Flask(__name__)
CORS(app)

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
<meta charset='utf-8'>
<meta name='viewport' content='width=device-width,initial-scale=1'>
<title>Wasel v4 Pro</title>
<style>
* { margin:0; padding:0; box-sizing:border-box; }
body { background:#0a0a0a; font-family:'Segoe UI',sans-serif; overflow:hidden; height:100vh; }
#container { position:relative; width:100vw; height:100vh; }
video { width:100%; height:100%; object-fit:cover; transform:scaleX(-1); }
#overlay { position:absolute; top:0; left:0; right:0; padding:20px 30px;
  background:linear-gradient(180deg,rgba(0,0,0,0.8) 0%,transparent 100%); }
#brand { color:#888; font-size:14px; letter-spacing:2px; text-transform:uppercase; }
#translation { color:#00ff88; font-size:48px; font-weight:700; margin-top:8px;
  text-shadow:0 2px 20px rgba(0,255,136,0.5); min-height:60px;
  transition:all 0.3s ease; }
#status { position:absolute; bottom:20px; left:30px; color:#555; font-size:13px; }
.pulse { animation: pulse 1.5s infinite; }
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.5} }
</style>
</head>
<body>
<div id='container'>
  <video id='cam' autoplay playsinline muted></video>
  <div id='overlay'>
    <div id='brand'>WASEL v4 PRO — GEMINI AI</div>
    <div id='translation' class='pulse'>Starting camera...</div>
  </div>
  <div id='status'>⚡ Gemini 2.0 Flash</div>
</div>
<canvas id='c' style='display:none'></canvas>
<script>
const video = document.getElementById('cam');
const canvas = document.getElementById('c');
const ctx = canvas.getContext('2d');
const translationEl = document.getElementById('translation');
const statusEl = document.getElementById('status');
const API = window.location.origin + '/translate';
let sending = false;

// Auto-start camera
navigator.mediaDevices.getUserMedia({video:{width:640,height:480,facingMode:'user'}})
.then(stream => {
  video.srcObject = stream;
  translationEl.textContent = 'Show a sign gesture...';
  translationEl.style.color = '#666';
  translationEl.classList.remove('pulse');
  setInterval(captureAndSend, 2000);
})
.catch(e => { translationEl.textContent = 'Camera error: ' + e.message; });

function captureAndSend() {
  if (sending) return;
  sending = true;
  canvas.width = 512; canvas.height = 384;
  ctx.drawImage(video, 0, 0, 512, 384);
  const data = canvas.toDataURL('image/jpeg', 0.7);
  statusEl.textContent = '⚡ Analyzing...';
  
  fetch(API, {
    method: 'POST',
    headers: {'Content-Type':'application/json'},
    body: JSON.stringify({image: data})
  })
  .then(r => r.json())
  .then(d => {
    const t = d.translation || '...';
    if (t === '...') {
      translationEl.textContent = 'Show a sign gesture...';
      translationEl.style.color = '#666';
    } else {
      translationEl.textContent = t;
      translationEl.style.color = '#00ff88';
    }
    statusEl.textContent = '⚡ Gemini 2.0 Flash — ' + new Date().toLocaleTimeString();
    sending = false;
  })
  .catch(e => { statusEl.textContent = 'Error: ' + e; sending = false; });
}
</script>
</body>
</html>
"""

@app.route('/')
def index():
    return Response(HTML_PAGE, mimetype='text/html')

@app.route('/translate', methods=['POST'])
def translate():
    try:
        data = request.json
        img_data = data['image'].split(',')[1]
        img_bytes = base64.b64decode(img_data)
        pil = Image.open(io.BytesIO(img_bytes))

        r = client.models.generate_content(
            model=MODEL,
            contents=[PROMPT, pil],
            config=types.GenerateContentConfig(max_output_tokens=20, temperature=0.1)
        )
        return jsonify({'translation': r.text.strip()})
    except Exception as e:
        return jsonify({'translation': f'Error: {str(e)[:40]}'})

# ─── Start server with public URL ───
def run_server():
    app.run(port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)

# Public URL
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    url = ngrok.connect(5000).public_url
else:
    # Fallback: use Colab's built-in proxy
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(5000)")

print("")
print("══════════════════════════════════════════════")
print("  🤟 Wasel v4 Pro — LIVE")
print("══════════════════════════════════════════════")
print(f"")
print(f"  🔗 افتح هذا الرابط:")
print(f"  {url}")
print(f"")
print(f"  📋 الكاميرا تبدأ تلقائياً — لا تضغط شيء!")
print(f"  📋 أعمل إشارة بيدك وشوف الترجمة فوق الفيديو")
print("══════════════════════════════════════════════")